In [ ]:
!pip install wfdb resampy ishneholterlib

In [ ]:
if not os.path.exists("/content/CMED-Mini-Project"):
 !git clone https://github.com/Anonymous0002396/CMED-Mini-Project.git /content/CMED-Mini-Project
else:
 %cd /content/CMED-Mini-Project
 !git pull

In [ ]:
import sys
from pathlib import Path

project_root = Path("/content/CMED-Mini-Project")
repo_root = project_root / "SSSD-ECG"

sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root / "src" / "ptb_xl"))
sys.path.insert(0, str(repo_root / "src" / "sssd"))

print(project_root)
print(repo_root.exists())

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from sklearn.metrics import roc_auc_score, average_precision_score

from pathlib import Path

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

In [ ]:

if not os.path.exists('/content/synthetic_ptbxl'): # Replace with the actual folder name inside the zip
 !cp "/content/drive/MyDrive/Dataset.zip" /content/synthetic_ptbxl.zip
 !unzip -oq /content/synthetic_ptbxl.zip -d /content/
 print("Extraction complete.")
else:
 print("Dataset already exists, skipping extraction.")

In [ ]:
if not os.path.exists('/content/ptbxl'): # Replace with the actual folder name inside the zip
 !cp "/content/drive/MyDrive/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.zip" /content/ptbxl.zip
 !unzip -oq /content/ptbxl.zip -d /content/
 print("Extraction complete.")
else:
 print("Dataset already exists, skipping extraction.")

### Set up paths

In [ ]:
target_fs = 100

# Raw PTB-XL folder after extraction
data_folder_ptb_xl = Path("/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1")

# Processed output folder used by `load_dataset(...)`
target_folder_ptb_xl = Path(f"/content/processed_ptb_xl_fs{target_fs}")

print("raw exists:", data_folder_ptb_xl.exists())
print("processed exists:", target_folder_ptb_xl.exists())

### Real PTB-XL preprocessing

In [ ]:
df_ptb_xl, lbl_itos_ptb_xl, mean_ptb_xl, std_ptb_xl = prepare_data_ptb_xl(
 data_path=data_folder_ptb_xl,
 min_cnt=0,
 target_fs=target_fs,
 channels=12,
 channel_stoi=channel_stoi_default,
 target_folder=target_folder_ptb_xl,
 recreate_data=True,
)

print(df_ptb_xl.shape)
print(type(lbl_itos_ptb_xl))
print(target_folder_ptb_xl)

In [ ]:
reformat_as_memmap(
 df_ptb_xl,
 target_folder_ptb_xl / "memmap.npy",
 data_folder=target_folder_ptb_xl,
 delete_npys=True
)

In [ ]:
df_mapped, lbl_itos_dict, mean, std = load_dataset(target_folder_ptb_xl)

print(type(df_mapped), df_mapped.shape)
print(type(lbl_itos_dict))
print(lbl_itos_dict.keys())

In [ ]:
label_key = "label_all" 
label_names = np.array(lbl_itos_dict[label_key])

print(label_key, len(label_names))
print(label_names[:30])

### Load synthetic dataset

In [ ]:
# Released synthetic PTB-XL dataset root after extraction
synthetic_root = Path("/content/Dataset")

syn_train_data_path = synthetic_root / "data" / "ptbxl_train_data.npy"
syn_val_data_path = synthetic_root / "data" / "ptbxl_validation_data.npy"
syn_test_data_path = synthetic_root / "data" / "ptbxl_test_data.npy"

syn_train_lbl_path = synthetic_root / "labels" / "ptbxl_train_labels.npy"
syn_val_lbl_path = synthetic_root / "labels" / "ptbxl_validation_labels.npy"
syn_test_lbl_path = synthetic_root / "labels" / "ptbxl_test_labels.npy"

for p in [
 syn_train_data_path, syn_val_data_path, syn_test_data_path,
 syn_train_lbl_path, syn_val_lbl_path, syn_test_lbl_path
]:
 print(p, "->", p.exists())

In [ ]:
syn_train_data = np.load(syn_train_data_path)
syn_train_labels = np.load(syn_train_lbl_path)

print("syn_train_data:", syn_train_data.shape, syn_train_data.dtype)
print("syn_train_labels:", syn_train_labels.shape, syn_train_labels.dtype)

### Label mapping + MI target creation

In [ ]:
mi_statement_names = [
 "IMI", "ASMI", "ILMI", "AMI", "ALMI",
 "INJAS", "LMI", "INJAL", "IPLMI", "IPMI",
 "INJIN", "PMI", "INJLA", "INJIL"
]

label_name_to_idx = {str(name): i for i, name in enumerate(label_names)}
mi_label_indices = [label_name_to_idx[name] for name in mi_statement_names]
mi_label_names = [str(label_names[i]) for i in mi_label_indices]

print("MI labels:", mi_label_names)

In [ ]:
def multilabel_to_binary_mi(y_multihot, mi_indices):
 return (y_multihot[:, mi_indices].sum(axis=1) > 0).astype(np.float32)

syn_train_mi = multilabel_to_binary_mi(syn_train_labels, mi_label_indices)


In [ ]:

# Skip if these objects are already loaded
syn_val_data = np.load(syn_val_data_path)
syn_val_labels = np.load(syn_val_lbl_path)
syn_test_data = np.load(syn_test_data_path)
syn_test_labels = np.load(syn_test_lbl_path)

syn_val_mi = multilabel_to_binary_mi(syn_val_labels, mi_label_indices)
syn_test_mi = multilabel_to_binary_mi(syn_test_labels, mi_label_indices)

print("Synthetic train MI prevalence:", syn_train_mi.mean(), syn_train_mi.sum(), len(syn_train_mi))
print("Synthetic val MI prevalence:", syn_val_mi.mean(), syn_val_mi.sum(), len(syn_val_mi))
print("Synthetic test MI prevalence:", syn_test_mi.mean(), syn_test_mi.sum(), len(syn_test_mi))

### Create multihot labels for the real PTB-XL metadata in the same 71-label space

In [ ]:
def multihot_encode(indices, num_classes):
 out = np.zeros(num_classes, dtype=np.float32)
 for i in indices:
 out[i] = 1.0
 return out

### Create binary MI targets for the real PTB-XL splits

In [ ]:
def row_multihot_to_binary_mi(row_vec, mi_indices):
 return float(row_vec[mi_indices].sum() > 0)

df_real = df_mapped.copy()

df_real["label_multihot"] = df_real[f"{label_key}_numeric"].apply(
 lambda x: multihot_encode(x, len(label_names))
)

df_real["label_mi"] = df_real["label_multihot"].apply(
 lambda x: row_multihot_to_binary_mi(x, mi_label_indices)
)

max_fold_id = df_real.strat_fold.max()

df_train_real = df_real[df_real.strat_fold < max_fold_id - 1].copy()
df_val_real = df_real[df_real.strat_fold == max_fold_id - 1].copy()
df_test_real = df_real[df_real.strat_fold == max_fold_id].copy()

print("Real train MI prevalence:", df_train_real["label_mi"].mean(), df_train_real["label_mi"].sum(), len(df_train_real))
print("Real val MI prevalence:", df_val_real["label_mi"].mean(), df_val_real["label_mi"].sum(), len(df_val_real))
print("Real test MI prevalence:", df_test_real["label_mi"].mean(), df_test_real["label_mi"].sum(), len(df_test_real))

### Memmap alignment block

In [ ]:
import pickle
import numpy as np

with open(target_folder_ptb_xl / "df_memmap.pkl", "rb") as f:
 df_memmap_meta = pickle.load(f)

df_memmap_meta = df_memmap_meta.copy()
df_memmap_meta["label_mi"] = df_real["label_mi"].values
df_memmap_meta["strat_fold"] = df_real["strat_fold"].values
df_memmap_meta["sex"] = df_real["sex"].values

max_fold_id = df_memmap_meta["strat_fold"].max()

df_train_real_mem = df_memmap_meta[df_memmap_meta["strat_fold"] < max_fold_id - 1].copy()
df_val_real_mem = df_memmap_meta[df_memmap_meta["strat_fold"] == max_fold_id - 1].copy()
df_test_real_mem = df_memmap_meta[df_memmap_meta["strat_fold"] == max_fold_id].copy()

print(len(df_train_real_mem), len(df_val_real_mem), len(df_test_real_mem))

In [ ]:

np.string_ = np.bytes_

In [ ]:
input_size = 1000

tfms_ptb_xl = ToTensor()

train_real_ds_full12 = TimeseriesDatasetCrops(
 df_train_real_mem,
 output_size=input_size,
 data_folder=target_folder_ptb_xl,
 chunk_length=0,
 min_chunk_length=input_size,
 stride=input_size,
 transforms=tfms_ptb_xl,
 annotation=False,
 col_lbl="label_mi",
 memmap_filename=target_folder_ptb_xl / "memmap.npy",
)

val_real_ds_full12 = TimeseriesDatasetCrops(
 df_val_real_mem,
 output_size=input_size,
 data_folder=target_folder_ptb_xl,
 chunk_length=0,
 min_chunk_length=input_size,
 stride=input_size,
 transforms=tfms_ptb_xl,
 annotation=False,
 col_lbl="label_mi",
 memmap_filename=target_folder_ptb_xl / "memmap.npy",
)

test_real_ds_full12 = TimeseriesDatasetCrops(
 df_test_real_mem,
 output_size=input_size,
 data_folder=target_folder_ptb_xl,
 chunk_length=0,
 min_chunk_length=input_size,
 stride=input_size,
 transforms=tfms_ptb_xl,
 annotation=False,
 col_lbl="label_mi",
 memmap_filename=target_folder_ptb_xl / "memmap.npy",
)

In [ ]:
sample = train_real_ds_full12[0]
print(type(sample))
print(sample[0].shape, sample[1])

In [ ]:
class SyntheticPTBXL8LeadMIDataset(Dataset):
 def __init__(self, signals, labels_binary):
 self.signals = np.asarray(signals, dtype=np.float32)
 self.labels = np.asarray(labels_binary, dtype=np.float32)

 def __len__(self):
 return len(self.signals)

 def __getitem__(self, idx):
 return {
 "signal": torch.tensor(self.signals[idx], dtype=torch.float32),
 "label": torch.tensor(self.labels[idx], dtype=torch.float32),
 }

train_syn_ds = SyntheticPTBXL8LeadMIDataset(syn_train_data, syn_train_mi)

In [ ]:
from torch.utils.data import Dataset

class TupleToDictWrapper(Dataset):
 def __init__(self, base_ds):
 self.base_ds = base_ds

 def __len__(self):
 return len(self.base_ds)

 def __getitem__(self, idx):
 x, y = self.base_ds[idx]

 x = torch.tensor(x, dtype=torch.float32) if not torch.is_tensor(x) else x.float()
 y = torch.tensor(y, dtype=torch.float32) if not torch.is_tensor(y) else y.float()

 return {"signal": x, "label": y}

In [ ]:
train_real_ds = TupleToDictWrapper(train_real_ds_full12)
val_real_ds = TupleToDictWrapper(val_real_ds_full12)
test_real_ds = TupleToDictWrapper(test_real_ds_full12)

batch_real = train_real_ds[0]
print(batch_real["signal"].shape, batch_real["label"], batch_real["signal"].dtype)

### Create augmented dataset

In [ ]:
from torch.utils.data import ConcatDataset

train_aug_ds = ConcatDataset([train_real_ds, train_syn_ds])

print("real train size:", len(train_real_ds))
print("synthetic train size:", len(train_syn_ds))
print("augmented train size:", len(train_aug_ds))

In [ ]:
from torch.utils.data import DataLoader

batch_size = 64

train_real_loader = DataLoader(
 train_real_ds,
 batch_size=batch_size,
 shuffle=True,
 num_workers=2,
 pin_memory=True,
)

train_aug_loader = DataLoader(
 train_aug_ds,
 batch_size=batch_size,
 shuffle=True,
 num_workers=2,
 pin_memory=True,
)

val_loader = DataLoader(
 val_real_ds,
 batch_size=batch_size,
 shuffle=False,
 num_workers=2,
 pin_memory=True,
)

test_loader = DataLoader(
 test_real_ds,
 batch_size=batch_size,
 shuffle=False,
 num_workers=2,
 pin_memory=True,
)

In [ ]:
print(df_test_real_mem["sex"].value_counts(dropna=False))

female_test_mask = df_test_real_mem["sex"].astype(str).str.lower().eq("female").to_numpy()
male_test_mask = df_test_real_mem["sex"].astype(str).str.lower().eq("male").to_numpy()

print("female:", female_test_mask.sum())
print("male:", male_test_mask.sum())
print("test len:", len(test_real_ds))

### Train model on only real ECG data

In [ ]:
import torch
import torch.nn as nn

class GRUBinaryMIClassifier(nn.Module):
 def __init__(
 self,
 num_signal_channels=12,
 gru_hidden_dim=128,
 gru_num_layers=2,
 mlp_hidden_dim=128,
 dropout=0.3,
 ):
 super().__init__()

 self.gru = nn.GRU(
 input_size=num_signal_channels,
 hidden_size=gru_hidden_dim,
 num_layers=gru_num_layers,
 batch_first=True,
 bidirectional=True,
 dropout=dropout if gru_num_layers > 1 else 0.0,
 )

 pooled_dim = 2 * gru_hidden_dim * 2 # avg + max pooling over bidirectional output

 self.fc1 = nn.Linear(pooled_dim, mlp_hidden_dim)
 self.relu = nn.ReLU()
 self.bn = nn.BatchNorm1d(mlp_hidden_dim)
 self.dropout = nn.Dropout(dropout)
 self.out = nn.Linear(mlp_hidden_dim, 1)

 def forward(self, signal):
 # signal: [B, 12, 1000] -> [B, 1000, 12]
 x = signal.transpose(1, 2)

 # x: [B, T, 2H]
 x, _ = self.gru(x)

 avg_pool = x.mean(dim=1)
 max_pool = x.max(dim=1).values
 feat = torch.cat([avg_pool, max_pool], dim=1)

 x = self.fc1(feat)
 x = self.relu(x)
 x = self.bn(x)
 x = self.dropout(x)
 logits = self.out(x).squeeze(1)

 return logits

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np

criterion = nn.BCEWithLogitsLoss()

def train_one_epoch(model, loader, optimizer, device):
 model.train()
 total_loss = 0.0
 n = 0

 for batch in loader:
 x = batch["signal"].to(device, non_blocking=True)
 y = batch["label"].to(device, non_blocking=True).float()

 optimizer.zero_grad()
 logits = model(x)
 loss = criterion(logits, y)
 loss.backward()
 optimizer.step()

 bs = x.size(0)
 total_loss += loss.item() * bs
 n += bs

 return total_loss / max(n, 1)

@torch.no_grad()
def evaluate(model, loader, device):
 model.eval()

 total_loss = 0.0
 n = 0

 all_logits = []
 all_probs = []
 all_targets = []

 for batch in loader:
 x = batch["signal"].to(device, non_blocking=True)
 y = batch["label"].to(device, non_blocking=True).float()

 logits = model(x)
 loss = criterion(logits, y)

 probs = torch.sigmoid(logits)

 bs = x.size(0)
 total_loss += loss.item() * bs
 n += bs

 all_logits.append(logits.detach().cpu())
 all_probs.append(probs.detach().cpu())
 all_targets.append(y.detach().cpu())

 all_logits = torch.cat(all_logits).numpy()
 all_probs = torch.cat(all_probs).numpy()
 all_targets = torch.cat(all_targets).numpy()

 return {
 "loss": total_loss / max(n, 1),
 "auroc": roc_auc_score(all_targets, all_probs),
 "auprc": average_precision_score(all_targets, all_probs),
 "probs": all_probs,
 "targets": all_targets,
 "logits": all_logits,
 }

In [ ]:
def fit_model(model, train_loader, val_loader, device, epochs=2, lr=1e-3):
 optimizer = torch.optim.Adam(model.parameters(), lr=lr)

 best_state = None
 best_val_auroc = -np.inf
 history = []

 for epoch in range(1, epochs + 1):
 train_loss = train_one_epoch(model, train_loader, optimizer, device)
 val_metrics = evaluate(model, val_loader, device)

 history.append({
 "epoch": epoch,
 "train_loss": train_loss,
 "val_loss": val_metrics["loss"],
 "val_auroc": val_metrics["auroc"],
 "val_auprc": val_metrics["auprc"],
 })

 print(
 f"Epoch {epoch}/{epochs} | "
 f"train_loss={train_loss:.4f} | "
 f"val_loss={val_metrics['loss']:.4f} | "
 f"val_auroc={val_metrics['auroc']:.4f} | "
 f"val_auprc={val_metrics['auprc']:.4f}"
 )

 if val_metrics["auroc"] > best_val_auroc:
 best_val_auroc = val_metrics["auroc"]
 best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

 model.load_state_dict(best_state)
 return model, history

In [ ]:
real_model = GRUBinaryMIClassifier(num_signal_channels=12).to(device)

real_model, real_history = fit_model(
 real_model,
 train_real_loader,
 val_loader,
 device,
 epochs=20,
 lr=1e-3,
)

### Train model on augmented ECG data

In [ ]:
aug_model = GRUBinaryMIClassifier(num_signal_channels=12).to(device)

aug_model, aug_history = fit_model(
 aug_model,
 train_aug_loader,
 val_loader,
 device,
 epochs=10,
 lr=1e-3,
)

### Comparing models

In [ ]:
real_test_metrics = evaluate(real_model, test_loader, device)
aug_test_metrics = evaluate(aug_model, test_loader, device)

print("REAL-ONLY TEST")
print({k: v for k, v in real_test_metrics.items() if k not in ["probs", "targets", "logits"]})

print("\nAUGMENTED TEST")
print({k: v for k, v in aug_test_metrics.items() if k not in ["probs", "targets", "logits"]})

### Female / male subgroup evaluation

In [ ]:
female_test_mask = (df_test_real_mem["sex"].to_numpy() == 1)
male_test_mask = (df_test_real_mem["sex"].to_numpy() == 0)

print("female:", female_test_mask.sum())
print("male:", male_test_mask.sum())

In [ ]:
def subgroup_metrics(targets, probs, mask):
 t = targets[mask]
 p = probs[mask]
 return {
 "n": len(t),
 "auroc": roc_auc_score(t, p),
 "auprc": average_precision_score(t, p),
 "prevalence": float(np.mean(t)),
 }

print("REAL-ONLY FEMALE")
print(subgroup_metrics(real_test_metrics["targets"], real_test_metrics["probs"], female_test_mask))

print("REAL-ONLY MALE")
print(subgroup_metrics(real_test_metrics["targets"], real_test_metrics["probs"], male_test_mask))

print("\nAUGMENTED FEMALE")
print(subgroup_metrics(aug_test_metrics["targets"], aug_test_metrics["probs"], female_test_mask))

print("AUGMENTED MALE")
print(subgroup_metrics(aug_test_metrics["targets"], aug_test_metrics["probs"], male_test_mask))

### Augmented only test

In [ ]:
train_syn_loader = DataLoader(
 train_syn_ds,
 batch_size=batch_size,
 shuffle=True,
 num_workers=2,
 pin_memory=True,
)

In [ ]:
syn_model = GRUBinaryMIClassifier(num_signal_channels=12).to(device)

syn_model, syn_history = fit_model(
 syn_model,
 train_syn_loader,
 val_loader, # still real validation
 device,
 epochs=10,
 lr=1e-3,
)

In [ ]:
syn_test_metrics = evaluate(syn_model, test_loader, device)

print("SYNTHETIC-ONLY TEST")
print({k: v for k, v in syn_test_metrics.items() if k not in ["probs", "targets", "logits"]})

### Testing smaller data augmentation 

In [ ]:
from torch.utils.data import Subset, ConcatDataset
import numpy as np

rng = np.random.default_rng(42)

n_syn = len(train_syn_ds)
idx_25 = rng.choice(n_syn, size=int(0.25 * n_syn), replace=False)
idx_50 = rng.choice(n_syn, size=int(0.50 * n_syn), replace=False)

train_syn_25_ds = Subset(train_syn_ds, idx_25)
train_syn_50_ds = Subset(train_syn_ds, idx_50)

train_aug_25_ds = ConcatDataset([train_real_ds, train_syn_25_ds])
train_aug_50_ds = ConcatDataset([train_real_ds, train_syn_50_ds])

print("real:", len(train_real_ds))
print("syn 25%:", len(train_syn_25_ds))
print("syn 50%:", len(train_syn_50_ds))
print("aug 25% total:", len(train_aug_25_ds))
print("aug 50% total:", len(train_aug_50_ds))

In [ ]:
train_aug_25_loader = DataLoader(
 train_aug_25_ds,
 batch_size=batch_size,
 shuffle=True,
 num_workers=2,
 pin_memory=True,
)

train_aug_50_loader = DataLoader(
 train_aug_50_ds,
 batch_size=batch_size,
 shuffle=True,
 num_workers=2,
 pin_memory=True,
)

In [ ]:
aug25_model = GRUBinaryMIClassifier(num_signal_channels=12).to(device)

aug25_model, aug25_history = fit_model(
 aug25_model,
 train_aug_25_loader,
 val_loader,
 device,
 epochs=10,
 lr=1e-3,
)

In [ ]:
aug50_model = GRUBinaryMIClassifier(num_signal_channels=12).to(device)

aug50_model, aug50_history = fit_model(
 aug50_model,
 train_aug_50_loader,
 val_loader,
 device,
 epochs=10,
 lr=1e-3,
)

In [ ]:
aug25_test_metrics = evaluate(aug25_model, test_loader, device)
aug50_test_metrics = evaluate(aug50_model, test_loader, device)

print("AUG 25% TEST")
print({k: v for k, v in aug25_test_metrics.items() if k not in ["probs", "targets", "logits"]})

print("\nAUG 50% TEST")
print({k: v for k, v in aug50_test_metrics.items() if k not in ["probs", "targets", "logits"]})